# 🪄 VietDocProof - Trợ lý AI Sửa Lỗi Chính Tả & Ngữ Pháp Tiếng Việt

Chào mừng bạn đến với công cụ tự động phát hiện và sửa lỗi văn bản Word bằng Trí tuệ nhân tạo. Công cụ này được thiết kế **dành cho tất cả mọi người**, kể cả khi bạn không am hiểu về lập trình.

---

In [ ]:
# @title ⚙️ BƯỚC 1: KHỞI TẠO HỆ THỐNG (Bấm nút Play ▶️ bên trái)
# @markdown Quá trình này sẽ tải lõi AI về máy chủ Google. Bạn chỉ cần chạy **1 lần duy nhất** cho mỗi phiên làm việc. Hãy kiên nhẫn đợi khoảng 1-2 phút nhé.

import os
import time
from IPython.display import clear_output

print("⏳ Đang tải hệ thống và các thư viện trí tuệ nhân tạo...")
print("☕ Vui lòng đợi trong giây lát...")

# Tải mã nguồn và cài đặt tĩnh lặng (giấu đi các log kỹ thuật rườm rà)
!git clone https://github.com/vonguyendang/VietDocProof.git &> /dev/null
%cd VietDocProof
!pip install -r requirements.txt &> /dev/null

import torch
import yaml

# Tối ưu hóa hệ thống để chạy nhanh gấp 8 lần
if os.path.exists('config/default.yaml'):
    with open('config/default.yaml', 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    config['model']['batch_size'] = 32
    with open('config/default.yaml', 'w', encoding='utf-8') as f:
        yaml.dump(config, f)

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('report', exist_ok=True)

clear_output()
if torch.cuda.is_available():
    print("✅ KHỞI TẠO THÀNH CÔNG! HỆ THỐNG ĐÃ SẴN SÀNG Ở TỐC ĐỘ CAO NHẤT (GPU).")
else:
    print("⚠️ HỆ THỐNG ĐÃ KHỞI TẠO, NHƯNG BẠN CHƯA BẬT CHẾ ĐỘ CHẠY NHANH (GPU)!")
    print("👉 Mẹo: Để chạy nhanh hơn, trên menu chọn: Runtime > Change runtime type > Chọn T4 GPU rồi chạy lại bước này.")

In [ ]:
# @title 🚀 BƯỚC 2: TẢI FILE LÊN & TỰ ĐỘNG SỬA LỖI
# @markdown 1. Chọn mức độ sửa đổi phù hợp ở danh sách bên dưới.
# @markdown 2. Bấm nút Play ▶️ bên trái.
# @markdown 3. Nhấp vào nút **Choose Files** hiện ra để chọn tài liệu Word từ máy tính của bạn.

Mức_Độ_Sửa_Lỗi = "An to\u00E0n (Ch\u1EC9 s\u1EEDa khi ch\u1EAFc ch\u1EAFn sai m\u1EDBi s\u1EEDa)" #@param ["An to\u00E0n (Ch\u1EC9 s\u1EEDa khi ch\u1EAFc ch\u1EAFn sai m\u1EDBi s\u1EEDa)", "G\u1EE3i \u00FD (\u0110\u00E1nh d\u1EA5u h\u1ED3ng c\u00E1c ch\u1ED7 kh\u00F4ng ch\u1EAFc ch\u1EAFn)", "M\u1EA1nh tay (AI t\u1EF1 \u0111\u1ED9ng bi\u00EAn t\u1EADp l\u1EA1i c\u00E2u)"]

mode_map = {
    "An to\u00E0n (Ch\u1EC9 s\u1EEDa khi ch\u1EAFc ch\u1EAFn sai m\u1EDBi s\u1EEDa)": "safe",
    "G\u1EE3i \u00FD (\u0110\u00E1nh d\u1EA5u h\u1ED3ng c\u00E1c ch\u1ED7 kh\u00F4ng ch\u1EAFc ch\u1EAFn)": "review",
    "M\u1EA1nh tay (AI t\u1EF1 \u0111\u1ED9ng bi\u00EAn t\u1EADp l\u1EA1i c\u00E2u)": "aggressive"
}
cli_mode = mode_map[Mức_Độ_Sửa_Lỗi]

import os
import shutil
from google.colab import files

# Xóa dữ liệu cũ của lần chạy trước (nếu có)
!rm -rf input output report KetQua_VietDocProof.zip
os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('report', exist_ok=True)

print("📥 Vui lòng bấm vào nút 'Choose Files' bên dưới để tải lên các file Word (.docx) cần sửa:\n")
uploaded = files.upload()

if not uploaded:
    print("\n❌ Bạn chưa chọn file nào. Vui lòng bấm Play chạy lại ô này khi đã sẵn sàng.")
else:
    for filename in uploaded.keys():
        if not filename.endswith('.docx'):
            print(f"\n⚠️ Đã bỏ qua file '{filename}' vì không phải định dạng Word (.docx).")
            continue
            
        print(f"\n🧠 AI đang đọc và phân tích lỗi cho file: '{filename}' (Mất khoảng vài chục giây)...")
        input_path = os.path.join('input', filename)
        shutil.move(filename, input_path)
        
        # Thực thi logic core. Giấu bớt các log thông thường nhưng nếu LỖI thì sẽ in ra đỏ chót để dễ bắt bệnh.
        !python cli.py --input "{input_path}" --output ./output --report ./report --mode {cli_mode} > /dev/null
        
        output_path = os.path.join('output', filename)
        if not os.path.exists(output_path):
            print(f"❌ Đã xảy ra lỗi không xác định khi xử lý file '{filename}'. Bạn xem dòng báo lỗi màu đỏ ở trên nếu có.")
            
    print("\n📦 Quá trình sửa lỗi hoàn tất. Đang đóng gói file Word và các file Báo Cáo chi tiết thành file ZIP...")
    !zip -q -r KetQua_VietDocProof.zip ./output ./report
    
    if os.path.exists("KetQua_VietDocProof.zip"):
        print("✅ Đã xong! Trình duyệt đang tự động tải file 'KetQua_VietDocProof.zip' chứa TOÀN BỘ KẾT QUẢ về máy tính...")
        print("👉 LƯU Ý: Nếu trình duyệt chặn tải xuống, bạn hãy nhìn sang menu bên trái > Chọn biểu tượng thư mục (Files) > Chuột phải vào file 'KetQua_VietDocProof.zip' và chọn Download.")
        files.download("KetQua_VietDocProof.zip")

In [ ]:
# @title 🌐 BƯỚC 3 (TÙY CHỌN): BẬT GIAO DIỆN WEB TRỰC QUAN
# @markdown Dành cho những ai thích có màn hình Giao diện Web (UI) xịn xò với các bảng báo cáo thống kê tỉ lệ sửa lỗi thay vì chỉ làm tự động như Bước 2.
# @markdown \nChạy ô này và chờ vài giây, nhấp vào **Public URL** (https://xxxxx.gradio.live) để mở sang tab mới.

print("⏳ Đang khởi tạo giao diện Web chuyên nghiệp...")
import logging
logging.getLogger('gradio').setLevel(logging.ERROR) # Giấu bớt log không cần thiết
!python wizard.py